# Adaptive Shield Integrations Overview (MVP)

This notebook builds an **integration-centric** overview:

1. Fetch all integrations in scope (paginated).
2. Fetch full security-check inventory in strict mode.
3. Fetch affected entities for failed, non-global checks.
4. Persist local daily snapshots for history cache (Parquet).
5. Render account -> integration -> check UI with collapsible history.
6. Export XLSX + CSV outputs.

Notes:
- History is built from local snapshots, not API event backfill.
- ServiceNow remains a stub and is empty by default.

In [ ]:
# Cell 1: Standalone Initialization (imports + helpers + config)
import json
import logging
import os
import re
import sys
from collections import defaultdict
from datetime import datetime, timezone
from html import escape
from pathlib import Path
from typing import Any

import pandas as pd
from dotenv import load_dotenv
from IPython.display import HTML, display


def env_bool(name: str, default: bool) -> bool:
    raw = os.getenv(name)
    if raw is None:
        return default
    return raw.strip().lower() in {'1', 'true', 'yes', 'y', 'on'}


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / '.env.example').exists() and (candidate / 'requirements.txt').exists():
            return candidate
    return cwd


ROOT_DIR = resolve_project_root()
SRC_DIR = ROOT_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from as_weekly_report import (
    AdaptiveShieldClient,
    CHECK_COLUMNS,
    FAILED_ENTITY_COLUMNS,
    INTEGRATION_COLUMNS,
    SNOW_COLUMNS,
    build_check_history_map,
    build_failed_entities_df,
    build_output_dirs,
    build_run_context,
    dedupe_daily_history,
    export_integration_overview,
    normalize_check_records,
    normalize_integration_records,
    normalize_status,
    read_snapshot_history,
    select_checks_inventory_strict,
    write_history_manifest,
    write_run_log,
    write_snapshot_parquet,
)


def _safe_json_load(raw: Any) -> Any:
    if raw is None:
        return None
    if isinstance(raw, (dict, list)):
        return raw
    text = str(raw).strip()
    if not text:
        return None
    try:
        return json.loads(text)
    except Exception:
        return raw


def _json_pretty_html(raw: Any) -> str:
    value = _safe_json_load(raw)
    if value in (None, '', [], {}):
        return '<span class="as-muted">N/A</span>'
    try:
        text = json.dumps(value, ensure_ascii=False, indent=2, default=str)
    except Exception:
        text = str(value)
    return f'<pre class="as-json">{escape(text)}</pre>'


def _is_populated(value: Any) -> bool:
    if value is None:
        return False
    if isinstance(value, str):
        return bool(value.strip())
    if isinstance(value, (list, dict, tuple, set)):
        return len(value) > 0
    return True


def _collect_matching_values(
    value: Any,
    *,
    key_pattern: re.Pattern[str],
    value_checker: Any = None,
    path: str = '',
) -> list[tuple[str, Any]]:
    results: list[tuple[str, Any]] = []

    if isinstance(value, dict):
        for key, nested_value in value.items():
            key_str = str(key)
            nested_path = f'{path}.{key_str}' if path else key_str
            if key_pattern.search(key_str):
                if value_checker is None or value_checker(nested_value):
                    results.append((nested_path, nested_value))
            results.extend(
                _collect_matching_values(
                    nested_value,
                    key_pattern=key_pattern,
                    value_checker=value_checker,
                    path=nested_path,
                )
            )
    elif isinstance(value, list):
        for index, nested_value in enumerate(value):
            nested_path = f'{path}[{index}]' if path else f'[{index}]'
            results.extend(
                _collect_matching_values(
                    nested_value,
                    key_pattern=key_pattern,
                    value_checker=value_checker,
                    path=nested_path,
                )
            )
    return results


def _entity_preview_model(entity: dict[str, Any]) -> dict[str, Any]:
    name = str(entity.get('entity_name') or '').strip()
    raw_sources = [
        _safe_json_load(entity.get('entity_raw_json')),
        _safe_json_load(entity.get('entity_usage_json')),
        _safe_json_load(entity.get('entity_extra_context_json')),
    ]

    emails: list[str] = []
    if '@' in name:
        emails.append(name)

    email_pattern = re.compile(r'email', re.I)
    for source in raw_sources:
        for _, email_value in _collect_matching_values(
            source,
            key_pattern=email_pattern,
            value_checker=lambda v: isinstance(v, str) and '@' in v,
        ):
            item = str(email_value).strip()
            if item and item not in emails:
                emails.append(item)

    date_pattern = re.compile(r'date|time|last|created|updated|seen|expiration|expiry', re.I)
    date_values: list[str] = []
    direct_expiry = entity.get('entity_dismiss_expiration_date')
    if _is_populated(direct_expiry):
        date_values.append(f'dismiss_expiration: {direct_expiry}')

    for source_name, source in (
        ('usage', raw_sources[1]),
        ('extra_context', raw_sources[2]),
        ('raw', raw_sources[0]),
    ):
        for path, value in _collect_matching_values(source, key_pattern=date_pattern):
            entry = f'{source_name}.{path}: {value}'
            if entry not in date_values:
                date_values.append(entry)

    return {
        'name': name,
        'emails': emails[:5],
        'dates': date_values[:8],
    }


def _entity_other_payload(entity: dict[str, Any]) -> dict[str, Any]:
    payload: dict[str, Any] = {}
    candidate = {
        'entity_type': entity.get('entity_type'),
        'entity_dismissed': entity.get('entity_dismissed'),
        'entity_dismissed_reason': entity.get('entity_dismissed_reason'),
        'entity_dismiss_expiration_date': entity.get('entity_dismiss_expiration_date'),
        'entity_extra_context': _safe_json_load(entity.get('entity_extra_context_json')),
        'entity_usage': _safe_json_load(entity.get('entity_usage_json')),
        'entity_raw': _safe_json_load(entity.get('entity_raw_json')),
    }
    for key, value in candidate.items():
        if _is_populated(value):
            payload[key] = value
    return payload


def _status_rank(status: str) -> int:
    order = {
        'failed': 0,
        'degraded': 1,
        'pending': 2,
        'drifted': 3,
        'passed': 4,
        'unknown': 5,
    }
    return order.get(status, 6)


def _style_badge(label: str, kind: str) -> str:
    return f'<span class="as-badge as-badge-{kind}">{escape(label)}</span>'


def render_integrations_overview_ui(
    checks_df: pd.DataFrame,
    entities_df: pd.DataFrame,
    history_map: dict[tuple[str, str, str], list[dict[str, Any]]],
) -> None:
    if checks_df.empty:
        display(HTML('<div style="padding:10px;border:1px solid #ddd;border-radius:8px;">No check records found for this run.</div>'))
        return

    entity_map: dict[tuple[str, str, str], list[dict[str, Any]]] = defaultdict(list)
    if not entities_df.empty:
        for entity in entities_df.to_dict('records'):
            key = (
                str(entity.get('account_id') or ''),
                str(entity.get('integration_id') or ''),
                str(entity.get('security_check_id') or ''),
            )
            entity_map[key].append(entity)

    checks = checks_df.to_dict('records')
    account_groups: dict[str, dict[str, Any]] = defaultdict(lambda: {'account_name': '', 'integrations': defaultdict(list)})

    for row in checks:
        account_id = str(row.get('account_id') or '')
        account_name = str(row.get('account_name') or '')
        integration_id = str(row.get('integration_id') or '')
        integration_name = str(row.get('integration_name') or row.get('saas_name') or '')
        integration_alias = str(row.get('integration_alias') or '')
        key = f'{integration_id}::{integration_name}::{integration_alias}'

        account_groups[account_id]['account_name'] = account_name
        account_groups[account_id]['integrations'][key].append(row)

    html_parts = [
        '<style>',
        '.as-ui {font-family: Arial, sans-serif; font-size: 13px; line-height: 1.4;}',
        '.as-title {font-size: 18px; font-weight: 700; margin-bottom: 10px;}',
        '.as-meta {color: #555; margin-bottom: 12px;}',
        '.as-account {border: 1px solid #d7d7d7; border-radius: 10px; margin-bottom: 12px; background: #fff;}',
        '.as-account > summary {cursor: pointer; padding: 10px 12px; font-weight: 700; background: #f9fafb; border-radius: 10px;}',
        '.as-account-body {padding: 8px 12px 12px 12px;}',
        '.as-integration {border: 1px solid #e5e7eb; border-radius: 8px; margin-bottom: 10px; background: #fff;}',
        '.as-integration > summary {cursor: pointer; padding: 8px 10px; font-weight: 700; background: #fcfcfd;}',
        '.as-check {border: 1px solid #ececec; border-radius: 8px; margin: 8px 0;}',
        '.as-check.failed {border-left: 5px solid #dc2626;}',
        '.as-check.passed {opacity: 0.95;}',
        '.as-check > summary {cursor: pointer; padding: 8px 10px; background: #fff; font-weight: 600;}',
        '.as-check-body {padding: 8px 10px 10px 10px;}',
        '.as-fold {margin-top: 6px; border: 1px dashed #d1d5db; border-radius: 6px; padding: 6px 8px; background: #fafafa;}',
        '.as-entities {margin-top: 8px;}',
        '.as-entity {margin-top: 6px; border: 1px solid #e5e7eb; border-radius: 6px; background: #fff; padding: 6px 8px;}',
        '.as-entity .label {font-weight: 600;}',
        '.as-history table {width: 100%; border-collapse: collapse; margin-top: 6px;}',
        '.as-history th, .as-history td {border: 1px solid #e5e7eb; padding: 4px 6px; text-align: left;}',
        '.as-history th {background: #f9fafb;}',
        '.as-badge {display: inline-block; margin-left: 6px; padding: 2px 6px; border-radius: 999px; font-size: 11px;}',
        '.as-badge-failed {background: #fee2e2; color: #991b1b;}',
        '.as-badge-passed {background: #dcfce7; color: #166534;}',
        '.as-badge-other {background: #e5e7eb; color: #374151;}',
        '.as-muted {color: #6b7280;}',
        '.as-json {margin: 6px 0 0 0; padding: 8px; background: #f3f4f6; border-radius: 6px; overflow-x: auto; white-space: pre-wrap;}',
        '</style>',
        '<div class="as-ui">',
        '<div class="as-title">Adaptive Shield Integration Overview</div>',
        f'<div class="as-meta">Integrations: {checks_df[["account_id", "integration_id"]].drop_duplicates().shape[0]} | Checks: {len(checks_df)} | Failed checks: {(checks_df["current_status"] == "failed").sum()}</div>',
    ]

    for account_id in sorted(account_groups.keys()):
        account_group = account_groups[account_id]
        account_name = account_group['account_name'] or 'Unknown Account'
        html_parts.append(f'<details class="as-account" open><summary>{escape(account_name)} ({escape(account_id)})</summary><div class="as-account-body">')

        integration_groups = account_group['integrations']
        sorted_integrations = sorted(
            integration_groups.items(),
            key=lambda item: item[0],
        )

        for integration_key, rows in sorted_integrations:
            rows_sorted = sorted(rows, key=lambda row: (_status_rank(str(row.get('current_status') or 'unknown')), str(row.get('security_check_name') or '')))
            integration_name = str(rows_sorted[0].get('integration_name') or rows_sorted[0].get('saas_name') or 'Unknown Integration')
            integration_alias = str(rows_sorted[0].get('integration_alias') or '')
            integration_id = str(rows_sorted[0].get('integration_id') or '')
            failed_count = sum(1 for row in rows_sorted if str(row.get('current_status') or '') == 'failed')

            summary = f'{integration_name} ({integration_alias}) [{integration_id}] - checks: {len(rows_sorted)}, failed: {failed_count}' if integration_alias else f'{integration_name} [{integration_id}] - checks: {len(rows_sorted)}, failed: {failed_count}'
            html_parts.append(f'<details class="as-integration" open><summary>{escape(summary)}</summary>')

            for row in rows_sorted:
                status = str(row.get('current_status') or 'unknown')
                check_name = str(row.get('security_check_name') or row.get('security_check_id') or 'Unknown Check')
                check_id = str(row.get('security_check_id') or '')
                badge_kind = 'failed' if status == 'failed' else ('passed' if status == 'passed' else 'other')
                badge = _style_badge(status.capitalize(), badge_kind)
                check_key = (account_id, integration_id, check_id)
                check_entities = entity_map.get(check_key, [])
                history_rows = history_map.get(check_key, [])
                open_attr = ' open' if status == 'failed' else ''

                html_parts.append(f'<details class="as-check {escape(status)}"{open_attr}>')
                html_parts.append(
                    '<summary>'
                    f'{escape(check_name)} {badge} '
                    f'<span class="as-muted">(affected: {escape(str(row.get("affected_entities_count") or 0))})</span>'
                    '</summary>'
                )
                html_parts.append('<div class="as-check-body">')
                html_parts.append(f'<div><strong>Impact:</strong> {escape(str(row.get("impact_level") or ""))}</div>')
                html_parts.append(f'<div><strong>Last change:</strong> {escape(str(row.get("last_change_datetime") or ""))}</div>')

                html_parts.append('<details class="as-fold"><summary>Check details</summary>')
                html_parts.append(f'<div>{escape(str(row.get("security_check_details") or ""))}</div></details>')

                html_parts.append('<details class="as-fold"><summary>Remediation</summary>')
                html_parts.append(f'<div>{escape(str(row.get("remediation_steps") or ""))}</div></details>')

                if status == 'failed':
                    html_parts.append('<details class="as-entities" open><summary>Affected entities</summary>')
                    if bool(row.get('is_global')):
                        html_parts.append('<div class="as-muted">Global check (entity-level list is not applicable).</div>')
                    elif check_entities:
                        for entity in check_entities:
                            preview = _entity_preview_model(entity)
                            html_parts.append('<div class="as-entity">')
                            if preview['name']:
                                html_parts.append(f'<div><span class="label">Name:</span> {escape(preview["name"])}</div>')
                            if preview['emails']:
                                html_parts.append(f'<div><span class="label">Email:</span> {escape(", ".join(preview["emails"]))}</div>')
                            if preview['dates']:
                                html_parts.append(f'<div><span class="label">Date:</span> {escape(" | ".join(preview["dates"]))}</div>')
                            other_payload = _entity_other_payload(entity)
                            if other_payload:
                                html_parts.append('<details><summary>Other populated fields</summary>')
                                html_parts.append(_json_pretty_html(other_payload))
                                html_parts.append('</details>')
                            html_parts.append('</div>')
                    else:
                        html_parts.append('<div class="as-muted">No affected entities returned.</div>')
                    html_parts.append('</details>')

                html_parts.append(f'<details class="as-history"><summary>History (daily snapshots): {len(history_rows)} records</summary>')
                if history_rows:
                    html_parts.append('<table><thead><tr><th>Snapshot Date (NY)</th><th>Run TS (UTC)</th><th>Status</th><th>Affected</th></tr></thead><tbody>')
                    for history_row in history_rows:
                        html_parts.append(
                            '<tr>'
                            f'<td>{escape(str(history_row.get("snapshot_date_ny") or ""))}</td>'
                            f'<td>{escape(str(history_row.get("run_ts_utc") or ""))}</td>'
                            f'<td>{escape(str(history_row.get("current_status") or ""))}</td>'
                            f'<td>{escape(str(history_row.get("affected_entities_count") or ""))}</td>'
                            '</tr>'
                        )
                    html_parts.append('</tbody></table>')
                else:
                    html_parts.append('<div class="as-muted">No snapshot history available for this check.</div>')
                html_parts.append('</details>')

                html_parts.append('</div></details>')

            html_parts.append('</details>')

        html_parts.append('</div></details>')

    html_parts.append('</div>')
    display(HTML(''.join(html_parts)))


load_dotenv(ROOT_DIR / '.env')

config = {
    'as_api_key': os.getenv('AS_API_KEY', '').strip(),
    'as_base_url': os.getenv('AS_BASE_URL', 'https://api.adaptive-shield.com').strip(),
    'as_account_ids': [
        account.strip()
        for account in os.getenv('AS_ACCOUNT_IDS', '').split(',')
        if account.strip()
    ],
    'lookback_days': int(os.getenv('LOOKBACK_DAYS', '3')),
    'output_root': os.getenv('OUTPUT_ROOT', 'output').strip(),
    'output_timezone': os.getenv('OUTPUT_TIMEZONE', 'America/New_York').strip(),
    'snapshot_granularity': os.getenv('SNAPSHOT_GRANULARITY', 'daily').strip(),
    'history_cache_enabled': env_bool('HISTORY_CACHE_ENABLED', True),
    'history_ui_lookback_days': int(os.getenv('HISTORY_UI_LOOKBACK_DAYS', '180')),
    'integration_ui_history_enabled': env_bool('INTEGRATION_UI_HISTORY_ENABLED', True),
    'export_xlsx': env_bool('EXPORT_XLSX', True),
    'export_csv': env_bool('EXPORT_CSV', True),
    'snow_enabled': env_bool('SNOW_ENABLED', False),
    'rate_limit_per_minute': int(os.getenv('RATE_LIMIT_PER_MINUTE', '90')),
    'request_timeout_seconds': int(os.getenv('REQUEST_TIMEOUT_SECONDS', '30')),
    'max_retries': int(os.getenv('MAX_RETRIES', '3')),
}

if not config['as_api_key']:
    raise ValueError('AS_API_KEY is required. Add it to .env before running.')

run_context = build_run_context(
    output_timezone=config['output_timezone'],
    snapshot_granularity=config['snapshot_granularity'],
)
output_dirs = build_output_dirs(ROOT_DIR / config['output_root'], run_context)

pipeline_errors: list[dict[str, Any]] = []

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
print('Configuration loaded:')
print({k: v for k, v in config.items() if k != 'as_api_key'})
print('Project root:', ROOT_DIR)
print('Run context:', run_context)
print('Output directories:', {k: str(v) for k, v in output_dirs.items()})

In [ ]:
# Cell 2: API Client Initialization
client = AdaptiveShieldClient(
    api_key=config['as_api_key'],
    base_url=config['as_base_url'],
    rate_limit_per_minute=config['rate_limit_per_minute'],
    timeout_seconds=config['request_timeout_seconds'],
    max_retries=config['max_retries'],
)

print('AdaptiveShieldClient initialized')

In [ ]:
# Cell 3: Get Accounts
accounts_records = []

if config['as_account_ids']:
    for account_id in config['as_account_ids']:
        accounts_records.append({'id': account_id, 'name': ''})
else:
    accounts_records = client.get_accounts()

accounts_df = pd.DataFrame(accounts_records)
if not accounts_df.empty:
    accounts_df = accounts_df.rename(columns={'id': 'account_id', 'name': 'account_name'})
    for column in ['account_id', 'account_name']:
        if column not in accounts_df.columns:
            accounts_df[column] = ''
    accounts_df = accounts_df[['account_id', 'account_name']]
else:
    accounts_df = pd.DataFrame(columns=['account_id', 'account_name'])

display(accounts_df.head(20))
print('Accounts found:', len(accounts_df))

In [ ]:
# Cell 4: Fetch Integrations (paginated)
integration_frames = []

for account in accounts_df.to_dict('records'):
    account_id = str(account.get('account_id') or '')
    account_name = str(account.get('account_name') or '')
    if not account_id:
        continue

    try:
        raw_integrations = client.get_integrations(account_id)
        normalized_df = normalize_integration_records(
            account_id=account_id,
            account_name=account_name,
            integration_records=raw_integrations,
            run_context=run_context,
        )
        integration_frames.append(normalized_df)
    except Exception as exc:
        pipeline_errors.append(
            {
                'stage': 'integrations',
                'account_id': account_id,
                'integration_id': None,
                'security_check_id': None,
                'message': str(exc),
            }
        )

if integration_frames:
    integrations_df = pd.concat(integration_frames, ignore_index=True, sort=False)
else:
    integrations_df = pd.DataFrame(columns=INTEGRATION_COLUMNS)

display(integrations_df.head(20))
print('Integrations found:', len(integrations_df))

In [ ]:
# Cell 5: Fetch Full Security Checks (strict mode)
check_frames = []
check_source_rows = []

for account in accounts_df.to_dict('records'):
    account_id = str(account.get('account_id') or '')
    account_name = str(account.get('account_name') or '')
    if not account_id:
        continue

    account_integrations_df = integrations_df[
        integrations_df['account_id'].astype(str) == account_id
    ].copy()
    integration_ids = (
        account_integrations_df['integration_id']
        .dropna()
        .astype(str)
        .drop_duplicates()
        .tolist()
    )

    try:
        raw_checks, source_used = select_checks_inventory_strict(
            account_id=account_id,
            integration_ids=integration_ids,
            fetch_account_checks=lambda account_id=account_id: client.get_security_checks_by_account(account_id),
            fetch_integration_checks=lambda integration_id, account_id=account_id: client.get_security_checks_by_integration(account_id, integration_id),
        )

        normalized_checks_df = normalize_check_records(
            account_id=account_id,
            account_name=account_name,
            check_records=raw_checks,
            integrations_df=account_integrations_df,
            run_context=run_context,
            strict_mapping=True,
        )
        check_frames.append(normalized_checks_df)

        check_source_rows.append(
            {
                'account_id': account_id,
                'source_used': source_used,
                'raw_count': len(raw_checks),
                'normalized_count': len(normalized_checks_df),
            }
        )
    except Exception as exc:
        pipeline_errors.append(
            {
                'stage': 'checks_inventory',
                'account_id': account_id,
                'integration_id': None,
                'security_check_id': None,
                'message': str(exc),
            }
        )
        raise

if check_frames:
    checks_df = pd.concat(check_frames, ignore_index=True, sort=False)
else:
    checks_df = pd.DataFrame(columns=CHECK_COLUMNS)

check_source_df = pd.DataFrame(check_source_rows)
display(check_source_df)
display(checks_df.head(20))
print('Security checks inventory rows:', len(checks_df))

In [ ]:
# Cell 6: Fetch Affected Entities for Failed Checks
failed_entity_records = []
affected_resolve_log_rows = []
affected_cache = {}

failed_checks_df = checks_df[
    checks_df['current_status'].fillna('').astype(str).str.lower() == 'failed'
].copy()

for check in failed_checks_df.to_dict('records'):
    account_id = str(check.get('account_id') or '')
    security_check_id = str(check.get('security_check_id') or '')
    integration_id = str(check.get('integration_id') or '')
    cache_key = (account_id, security_check_id)

    if bool(check.get('is_global')):
        affected_resolve_log_rows.append(
            {
                'account_id': account_id,
                'integration_id': integration_id,
                'security_check_id': security_check_id,
                'resolve_status': 'global',
                'message': 'Global check - entity list is not applicable',
            }
        )
        continue

    if not account_id or not security_check_id:
        pipeline_errors.append(
            {
                'stage': 'affected_entities',
                'account_id': account_id,
                'integration_id': integration_id,
                'security_check_id': security_check_id,
                'message': 'Missing account_id or security_check_id',
            }
        )
        affected_resolve_log_rows.append(
            {
                'account_id': account_id,
                'integration_id': integration_id,
                'security_check_id': security_check_id,
                'resolve_status': 'unresolved',
                'message': 'Missing account_id or security_check_id',
            }
        )
        continue

    entities = []
    if cache_key in affected_cache:
        entities = affected_cache[cache_key]
    else:
        try:
            entities = client.get_affected_entities(account_id, security_check_id)
            affected_cache[cache_key] = entities
        except Exception as exc:
            pipeline_errors.append(
                {
                    'stage': 'affected_entities',
                    'account_id': account_id,
                    'integration_id': integration_id,
                    'security_check_id': security_check_id,
                    'message': str(exc),
                }
            )
            affected_resolve_log_rows.append(
                {
                    'account_id': account_id,
                    'integration_id': integration_id,
                    'security_check_id': security_check_id,
                    'resolve_status': 'unresolved',
                    'message': 'Affected API failed',
                }
            )
            continue

    for entity in entities:
        failed_entity_records.append(
            {
                'account_id': check.get('account_id'),
                'account_name': check.get('account_name'),
                'integration_id': check.get('integration_id'),
                'integration_name': check.get('integration_name'),
                'integration_alias': check.get('integration_alias'),
                'saas_name': check.get('saas_name'),
                'security_check_id': check.get('security_check_id'),
                'security_check_name': check.get('security_check_name'),
                'current_status': check.get('current_status'),
                'entity': entity,
            }
        )

    affected_resolve_log_rows.append(
        {
            'account_id': account_id,
            'integration_id': integration_id,
            'security_check_id': security_check_id,
            'resolve_status': 'fetched',
            'message': f'Fetched {len(entities)} entities',
        }
    )

entities_df = build_failed_entities_df(failed_entity_records, run_context)
if entities_df.empty:
    entities_df = pd.DataFrame(columns=FAILED_ENTITY_COLUMNS)

affected_resolve_log_df = pd.DataFrame(affected_resolve_log_rows)
display(entities_df.head(20))
display(affected_resolve_log_df.head(20))
print('Failed entity rows:', len(entities_df))

In [ ]:
# Cell 7: Persist Daily Snapshots (Parquet)
history_paths = {
    'check_snapshot_path': '',
    'failed_entities_snapshot_path': '',
}

if config['history_cache_enabled']:
    try:
        check_snapshot_path = write_snapshot_parquet(
            checks_df,
            output_dirs['history'],
            dataset='check_snapshot',
            run_context=run_context,
            file_prefix='check_snapshot',
        )
        failed_entities_snapshot_path = write_snapshot_parquet(
            entities_df,
            output_dirs['history'],
            dataset='failed_entities_snapshot',
            run_context=run_context,
            file_prefix='failed_entities_snapshot',
        )
        history_paths['check_snapshot_path'] = str(check_snapshot_path)
        history_paths['failed_entities_snapshot_path'] = str(failed_entities_snapshot_path)
    except Exception as exc:
        pipeline_errors.append(
            {
                'stage': 'history_write',
                'account_id': None,
                'integration_id': None,
                'security_check_id': None,
                'message': str(exc),
            }
        )
        print('History write skipped:', exc)
else:
    print('History cache is disabled by configuration.')

display(pd.DataFrame([history_paths]))

In [ ]:
# Cell 8: Build History View from Local Snapshots
history_raw_df = pd.DataFrame()
history_daily_df = pd.DataFrame()
history_map = {}

if config['integration_ui_history_enabled'] and config['history_cache_enabled']:
    try:
        history_raw_df = read_snapshot_history(
            output_dirs['history'],
            dataset='check_snapshot',
            lookback_days=config['history_ui_lookback_days'],
        )
        if not history_raw_df.empty:
            history_daily_df = dedupe_daily_history(
                history_raw_df,
                key_columns=['account_id', 'integration_id', 'security_check_id'],
            )
            history_map = build_check_history_map(history_daily_df)
    except Exception as exc:
        pipeline_errors.append(
            {
                'stage': 'history_read',
                'account_id': None,
                'integration_id': None,
                'security_check_id': None,
                'message': str(exc),
            }
        )
        print('History read skipped:', exc)
else:
    print('History UI is disabled by configuration.')

display(history_daily_df.head(20) if not history_daily_df.empty else history_daily_df)
print('History rows (daily deduped):', len(history_daily_df))

In [ ]:
# Cell 9: Render Integrations Overview UI
render_integrations_overview_ui(checks_df, entities_df, history_map)

In [ ]:
# Cell 10: ServiceNow Stub (empty by default)
snow_df = pd.DataFrame(columns=['integration_id', *SNOW_COLUMNS])
summary_with_snow_df = integrations_df.copy()

for column in SNOW_COLUMNS:
    if column not in summary_with_snow_df.columns:
        summary_with_snow_df[column] = None

if config['snow_enabled']:
    print('SNOW_ENABLED=true, but ServiceNow integration is not implemented in this notebook version.')

display(snow_df.head(20))
display(summary_with_snow_df.head(20))

In [ ]:
# Cell 11: Export Files + Logs
ts = run_context['run_ts_utc'].replace('-', '').replace(':', '').replace('T', '_').replace('Z', '')

errors_df = pd.DataFrame(
    pipeline_errors,
    columns=['stage', 'account_id', 'integration_id', 'security_check_id', 'message'],
)

export_paths = export_integration_overview(
    summary_df=summary_with_snow_df,
    checks_df=checks_df,
    entities_df=entities_df,
    errors_df=errors_df,
    overview_dir=output_dirs['overview'],
    ts=ts,
    export_xlsx=config['export_xlsx'],
    export_csv=config['export_csv'],
)

quality_rows = []
quality_rows.append(
    {
        'check': 'missing_integration_columns',
        'value': int(len([col for col in INTEGRATION_COLUMNS if col not in summary_with_snow_df.columns])),
        'details': ', '.join([col for col in INTEGRATION_COLUMNS if col not in summary_with_snow_df.columns]),
    }
)
quality_rows.append(
    {
        'check': 'missing_check_columns',
        'value': int(len([col for col in CHECK_COLUMNS if col not in checks_df.columns])),
        'details': ', '.join([col for col in CHECK_COLUMNS if col not in checks_df.columns]),
    }
)
quality_rows.append(
    {
        'check': 'missing_failed_entity_columns',
        'value': int(len([col for col in FAILED_ENTITY_COLUMNS if col not in entities_df.columns])),
        'details': ', '.join([col for col in FAILED_ENTITY_COLUMNS if col not in entities_df.columns]),
    }
)
quality_rows.append(
    {
        'check': 'duplicate_check_keys',
        'value': int(
            checks_df.duplicated(subset=['account_id', 'integration_id', 'security_check_id']).sum()
            if not checks_df.empty
            else 0
        ),
        'details': 'subset=[account_id, integration_id, security_check_id]',
    }
)

quality_report_df = pd.DataFrame(quality_rows)
quality_report_path = output_dirs['log'] / f'quality_report_{ts}.csv'
quality_report_df.to_csv(quality_report_path, index=False)

history_manifest_path = None
if config['history_cache_enabled']:
    history_manifest_path = write_history_manifest(
        output_dirs['history'],
        run_context=run_context,
        payload={
            'run_context': run_context,
            'history_paths': history_paths,
            'checks_rows': len(checks_df),
            'failed_entities_rows': len(entities_df),
            'history_daily_rows': len(history_daily_df),
            'error_rows': len(errors_df),
        },
    )

run_log_path = write_run_log(
    output_dirs['log'],
    run_context=run_context,
    payload={
        'run_context': run_context,
        'config': {k: v for k, v in config.items() if k != 'as_api_key'},
        'counts': {
            'accounts': len(accounts_df),
            'integrations': len(integrations_df),
            'checks': len(checks_df),
            'failed_entities': len(entities_df),
            'errors': len(errors_df),
        },
        'export_paths': export_paths,
        'quality_report_path': str(quality_report_path),
        'history_manifest_path': str(history_manifest_path) if history_manifest_path else '',
    },
)

print('Export paths:')
for key, value in export_paths.items():
    print(f'- {key}: {value}')
print('Quality report:', quality_report_path)
print('Run log:', run_log_path)
if history_manifest_path:
    print('History manifest:', history_manifest_path)